# AquaInsight — Task 2: Skewness Reduction Study — Box-Cox and Yeo-Johnson

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


# Task 2 — In-Depth Skewness Reduction Study

### Internship requirement
Apply **Box-Cox and Yeo-Johnson transformations** to non-normally distributed features, evaluate pre- and post-transformation **skewness and kurtosis**, and assess improvement in distributional symmetry.

### Steps
1. Select water-quality characteristics with enough observations.
2. Calculate original skewness and kurtosis.
3. Apply Yeo-Johnson to support zero/negative values.
4. Apply Box-Cox only to strictly positive values.
5. Compare pre- and post-transformation metrics.
6. Identify which transformation produces the strongest reduction in skewness.

In [6]:
import numpy as np
import pandas as pd
from scipy import stats


def transform_skewness(vals):
    s = pd.to_numeric(pd.Series(vals), errors="coerce").dropna()
    if s.empty:
        return {
            "original_skew": np.nan,
            "yeo_johnson_skew": np.nan,
            "box_cox_skew": np.nan,
            "original_kurtosis": np.nan,
            "yeo_johnson_kurtosis": np.nan,
            "box_cox_kurtosis": np.nan,
            "yeo_johnson_lambda": np.nan,
            "box_cox_lambda": np.nan,
        }

    original_skew = s.skew()
    original_kurtosis = s.kurt()

    try:
        yj_vals, yj_lambda = stats.yeojohnson(s)
        yeo_johnson_skew = pd.Series(yj_vals).skew()
        yeo_johnson_kurtosis = pd.Series(yj_vals).kurt()
    except Exception:
        yj_vals = s
        yj_lambda = np.nan
        yeo_johnson_skew = original_skew
        yeo_johnson_kurtosis = original_kurtosis

    try:
        bc_vals, bc_lambda = stats.boxcox(s)
        box_cox_skew = pd.Series(bc_vals).skew()
        box_cox_kurtosis = pd.Series(bc_vals).kurt()
    except Exception:
        bc_lambda = np.nan
        box_cox_skew = original_skew
        box_cox_kurtosis = original_kurtosis

    return {
        "original_skew": original_skew,
        "yeo_johnson_skew": yeo_johnson_skew,
        "box_cox_skew": box_cox_skew,
        "original_kurtosis": original_kurtosis,
        "yeo_johnson_kurtosis": yeo_johnson_kurtosis,
        "box_cox_kurtosis": box_cox_kurtosis,
        "yeo_johnson_lambda": yj_lambda,
        "box_cox_lambda": bc_lambda,
    }


candidate_chars = [
    "Turbidity",
    "Nitrate",
    "Total Nitrogen, mixed forms",
    "Total Phosphorus, mixed forms",
    "Specific conductance"
]

rows = []
for char in candidate_chars:
    vals = df.loc[df["CharacteristicName"].eq(char), "ResultValue"].dropna()
    if len(vals) >= 100:
        result = transform_skewness(vals)
        result["characteristic"] = char
        result["n"] = len(vals)
        rows.append(result)

skew_results = pd.DataFrame(rows)[[
    "characteristic", "n", "original_skew", "yeo_johnson_skew",
    "box_cox_skew", "original_kurtosis",
    "yeo_johnson_kurtosis", "box_cox_kurtosis",
    "yeo_johnson_lambda", "box_cox_lambda"
]]
display(skew_results.round(4))


,characteristic,n,original_skew,yeo_johnson_skew,box_cox_skew,original_kurtosis,yeo_johnson_kurtosis,box_cox_kurtosis,yeo_johnson_lambda,box_cox_lambda
0,Turbidity,6605,35.8653,0.2173,35.8653,1646.1365,-0.5377,1646.1365,-0.8651,NaN
1,Nitrate,2121,2.9143,0.8105,0.1456,7.9751,-0.7963,-1.1023,-2.3709,-0.2079
2,"Total Nitrogen, mixed forms",7333,4.6941,0.4266,-0.0719,23.1240,-0.2932,1.1805,-2.8480,-0.2141
3,"Total Phosphorus, mixed forms",5424,8.2248,0.9595,-0.0053,111.5617,0.1571,0.2785,-46.1841,-0.4404
4,Specific conductance,8963,6.5432,-0.2732,-0.1711,84.2950,6.6480,7.5611,-0.2785,-0.0921


## Task 2 — Conclusion

The analysis above completes the requested **Skewness Reduction Study — Box-Cox and Yeo-Johnson** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.